In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine

# 1) Cargar .env de forma robusta
env_path = find_dotenv(usecwd=True)  # busca .env desde la cwd hacia arriba
if not env_path:
    raise FileNotFoundError("No se encontró .env. Verifica ubicación/nombre.")
load_dotenv(env_path, override=True)

# 2) Leer variables y validar
required = ["MYSQL_HOST", "MYSQL_PORT", "MYSQL_DB", "MYSQL_USER", "MYSQL_PASSWORD"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(f"Faltan variables en .env: {missing}")

host = os.getenv("MYSQL_HOST")
port = int(os.getenv("MYSQL_PORT", "3306"))
db   = os.getenv("MYSQL_DB")
user = os.getenv("MYSQL_USER")
pwd  = os.getenv("MYSQL_PASSWORD")

print("Usando host:", host, "| db:", db, "| user:", user)  # no imprimas la clave

# 3) Crear engine y probar
engine = create_engine(
    f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}?charset=utf8mb4",
    pool_pre_ping=True
)

df = pd.read_sql("SELECT COUNT(*) AS filas FROM FactVentasDetalle", engine)
df


# Ejemplo: traer ventas por día (top 10)
q = """
SELECT f.Fecha, SUM(v.Total) AS VentasTotal
FROM FactVentasDetalle v
JOIN DimFecha f ON f.FechaID = v.FechaFacturaID
GROUP BY f.Fecha
ORDER BY f.Fecha DESC
LIMIT 10;
"""
preview = pd.read_sql(q, engine)
preview.head()




Usando host: 127.0.0.1 | db: dw_dev | user: root


,Fecha,VentasTotal
0,2025-06-30,220.66
1,2025-06-29,109.74
2,2025-06-28,540.38
3,2025-06-27,428.46
4,2025-06-26,318.54


In [2]:
#1) Helpers (reutilizables)
import pandas as pd
from sqlalchemy import text

def q(sql, params=None):
    """Ejecuta SQL y devuelve DataFrame."""
    return pd.read_sql(text(sql), engine, params=params or {})

def to_csv(df, path):
    df.to_csv(path, index=False)
    print(f"Guardado: {path}")

In [3]:
#KPI básicos (Ventas, Egresos, Margen) por día

sql_margen_dia = """
WITH v AS (
  SELECT v.FechaFacturaID, SUM(v.SinImpuesto) AS Ventas
  FROM FactVentasDetalle v
  GROUP BY v.FechaFacturaID
),
e AS (
  SELECT e.FechaFacturaID, SUM(e.SinImpuesto) AS Egresos
  FROM FactEgresosDetalle e
  GROUP BY e.FechaFacturaID
)
SELECT
  f.Fecha,
  COALESCE(v.Ventas,0)  AS Ventas,
  COALESCE(e.Egresos,0) AS Egresos,
  COALESCE(v.Ventas,0) - COALESCE(e.Egresos,0) AS Margen
FROM DimFecha f
LEFT JOIN v ON v.FechaFacturaID = f.FechaID
LEFT JOIN e ON e.FechaFacturaID = f.FechaID
WHERE f.Fecha BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY f.Fecha;
"""
df_margen_dia = q(sql_margen_dia)
df_margen_dia.head()


,Fecha,Ventas,Egresos,Margen
0,2025-01-01,51.00,61.2,-10.20
1,2025-01-02,78.70,94.2,-15.50
2,2025-01-03,108.10,129.0,-20.90
3,2025-01-04,139.20,165.6,-26.40
4,2025-01-05,29.25,31.5,-2.25


In [4]:
#Top Productos por ventas y marge

sql_top_prod = """
WITH vent AS (
  SELECT ProductoID, SUM(SinImpuesto) AS Ventas
  FROM FactVentasDetalle
  GROUP BY ProductoID
),
eg AS (
  SELECT ProductoID, SUM(SinImpuesto) AS Egresos
  FROM FactEgresosDetalle
  GROUP BY ProductoID
)
SELECT
  p.ProductoID,
  p.NomProducto,
  COALESCE(vent.Ventas,0)  AS Ventas,
  COALESCE(eg.Egresos,0)   AS Egresos,
  COALESCE(vent.Ventas,0) - COALESCE(eg.Egresos,0) AS Margen
FROM DimProducto p
LEFT JOIN vent ON vent.ProductoID = p.ProductoID
LEFT JOIN eg   ON eg.ProductoID   = p.ProductoID
ORDER BY Margen DESC
LIMIT 20;
"""
df_top_prod = q(sql_top_prod)
df_top_prod


,ProductoID,NomProducto,Ventas,Egresos,Margen
0,2,Producto 002,476.00,460.8,15.20
1,1,Producto 001,211.00,198.0,13.00
2,6,Producto 006,155.75,148.5,7.25
3,11,Producto 011,168.50,163.8,4.70
4,36,Producto 036,232.25,229.5,2.75
5,16,Producto 016,181.25,179.1,2.15
6,12,Producto 012,340.00,338.4,1.60
7,21,Producto 021,194.00,194.4,-0.40
8,17,Producto 017,365.50,367.2,-1.70
9,26,Producto 026,206.75,209.7,-2.95


In [5]:
#Margen por Evento y Organizador (sin duplicar)

#Como un evento puede tener varios organizadores, si sumas directo por el bridge duplicas los importes.
#La forma correcta es prorratear por el número de organizadores del evento.

sql_margen_org = """
WITH cnt_org AS (
  SELECT EventoID, COUNT(*) AS n
  FROM eventosOrganizador_Detalle
  GROUP BY EventoID
),
v AS (
  SELECT EventoID, SUM(SinImpuesto) AS Ventas
  FROM FactVentasDetalle
  GROUP BY EventoID
),
e AS (
  SELECT EventoID, SUM(SinImpuesto) AS Egresos
  FROM FactEgresosDetalle
  GROUP BY EventoID
)
SELECT
  o.OrganizadorID,
  d.Nombre AS Organizador,
  SUM( COALESCE(v.Ventas,0)  / NULLIF(c.n,0) ) AS VentasAjustadas,
  SUM( COALESCE(e.Egresos,0) / NULLIF(c.n,0) ) AS EgresosAjustados,
  SUM( COALESCE(v.Ventas,0)  / NULLIF(c.n,0)
     - COALESCE(e.Egresos,0) / NULLIF(c.n,0) ) AS MargenAjustado
FROM eventosOrganizador_Detalle o
JOIN cnt_org c ON c.EventoID = o.EventoID
LEFT JOIN v     ON v.EventoID = o.EventoID
LEFT JOIN e     ON e.EventoID = o.EventoID
LEFT JOIN DimOrganizadores d ON d.OrganizadorID = o.OrganizadorID
GROUP BY o.OrganizadorID, d.Nombre
ORDER BY MargenAjustado DESC;
"""
df_margen_org = q(sql_margen_org)
df_margen_org.head()


,OrganizadorID,Organizador,VentasAjustadas,EgresosAjustados,MargenAjustado
0,7,Organizador 07,550.125,627.75,-77.625
1,2,Organizador 02,588.375,668.25,-79.875
2,12,Organizador 12,554.375,641.25,-86.875
3,17,Organizador 17,592.625,681.75,-89.125
4,8,Organizador 08,1134.750,1315.80,-181.050


In [6]:
#5) Precio promedio de ticket por evento

sql_precio_evento = """
SELECT
  ev.EventoID,
  ev.NombreEvento,
  SUM(v.Total)/NULLIF(SUM(v.CantidadProd),0) AS PrecioPromedio,
  SUM(v.CantidadProd) AS Unidades,
  SUM(v.SinImpuesto) AS Ventas
FROM FactVentasDetalle v
JOIN DimEventos ev ON ev.EventoID = v.EventoID
GROUP BY ev.EventoID, ev.NombreEvento
ORDER BY Ventas DESC
LIMIT 30;
"""
df_precio_evento = q(sql_precio_evento)
df_precio_evento.head()


,EventoID,NombreEvento,PrecioPromedio,Unidades,Ventas
0,20,Evento 20,88.016,25.0,1864.75
1,15,Evento 15,83.002,25.0,1758.50
2,10,Evento 10,77.986,25.0,1652.25
3,5,Evento 05,72.972,25.0,1546.00
4,19,Evento 19,87.055,20.0,1475.50


In [7]:
#6) Guardar resultados
to_csv(df_margen_dia,   "out/margen_por_dia.csv")
to_csv(df_top_prod,     "out/top_productos.csv")
to_csv(df_margen_org,   "out/margen_por_organizador.csv")
to_csv(df_precio_evento,"out/precio_promedio_evento.csv")


Guardado: out/margen_por_dia.csv
Guardado: out/top_productos.csv
Guardado: out/margen_por_organizador.csv
Guardado: out/precio_promedio_evento.csv
